# Final project

Team1: Karl Prokop and Amanda Christianson

## Checkpoint 1: Sentinel-2 data selection and retrieval 

### Importing libraries and configuration of OAuth2 Client Credentials

In [3]:
# Importing libraries
import requests
import os
import json
from datetime import datetime
import zipfile
from pathlib import Path

# For visualization later
import matplotlib.pyplot as plt

In [4]:
# Configuration of credentials

# Direct assignment
COPERNICUS_CLIENT_ID = os.getenv('COPERNICUS_CLIENT_ID', 'sh-c5dcc309-63e8-491b-8c97-47925cbe91ea')
COPERNICUS_CLIENT_SECRET = os.getenv('COPERNICUS_CLIENT_SECRET', 'uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL')

# Karl's key
# sh-488cead5-2fcf-4384-ae68-0929267aa550
# 3fPSrDkk2UdHWozCmIFnGHxgAnyZc6jj

# Amanda's key
# sh-c5dcc309-63e8-491b-8c97-47925cbe91ea
# uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL

# Copernicus Dataspace API endpoints
AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SEARCH_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD_URL = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

print("✓ Credentials configured")

✓ Credentials configured


In [5]:
def get_access_token(client_id, client_secret):
    """
    Get OAuth2 access token from Copernicus Dataspace using Client Credentials flow.
    
    This is the recommended method for server-to-server authentication and HPC jobs.
    
    Parameters:
    -----------
    client_id : str
        OAuth2 Client ID (starts with 'sh-')
    client_secret : str
        OAuth2 Client Secret
    
    Returns:
    --------
    str : Access token if successful, None otherwise
    """
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    
    try:
        response = requests.post(AUTH_URL, data=data, timeout=30)
        response.raise_for_status()
        token_data = response.json()
        
        # Extract token and expiration
        access_token = token_data["access_token"]
        expires_in = token_data.get("expires_in", 3600)
        
        print(f"✓ Token obtained (valid for {expires_in//60} minutes)")
        return access_token
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            print("❌ Authentication failed: Invalid credentials")
            print("   ✗ Check your CLIENT_ID and CLIENT_SECRET")
            print("   ✗ CLIENT_ID should start with 'sh-'")
        else:
            print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
        return None
        
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        return None

# Get access token
print("Authenticating with Copernicus Dataspace...")
access_token = get_access_token(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)

if access_token:
    print("✓ Successfully authenticated")
    headers = {"Authorization": f"Bearer {access_token}"}
else:
    print("❌ Authentication failed.")
    print("\nTroubleshooting:")
    print("1. Check environment variables are set:")
    print(f"   COPERNICUS_CLIENT_ID = {COPERNICUS_CLIENT_ID[:15]}...")
    print(f"   COPERNICUS_CLIENT_SECRET = {COPERNICUS_CLIENT_SECRET[:15]}...")
    print("2. Verify credentials in Copernicus Dashboard")
    print("3. See COPERNICUS_SETUP.md for detailed instructions")
    headers = None

Authenticating with Copernicus Dataspace...
✓ Token obtained (valid for 30 minutes)
✓ Successfully authenticated


### Define Search Parameters (region of interest and date range)

In [6]:
# Define your Region of Interest (ROI) as a bounding box
# Format: POLYGON((lon lat, lon lat, ...))
# Note: You will be selecting an MGRS tile within this region

# Bounding box coordinates [min_lon, min_lat, max_lon, max_lat]
# Middle of Sweden (Uppsala to Umeå), Sweden
min_lon, min_lat = 12.5, 60
max_lon, max_lat = 17.5, 63.5

# Create WKT POLYGON for API query
roi_polygon = f"POLYGON(({min_lon} {min_lat},{max_lon} {min_lat},{max_lon} {max_lat},{min_lon} {max_lat},{min_lon} {min_lat}))"

# Define date range
start_date = '2018-03-01T00:00:00.000Z'
end_date = '2018-10-31T23:59:59.999Z'

# Maximum cloud cover percentage (30% to get good data availability)
max_cloud_cover = 30

print(f"Search Parameters:")
print(f"  Region: Bavarian region (Central Europe with CORINE coverage)")
print(f"  Bounding Box: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"  Date Range: {start_date[:10]} to {end_date[:10]}")
print(f"  Max Cloud Cover: {max_cloud_cover}%")
print(f"\nNote: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).")
print(f"Your search will find all tiles intersecting this region.")

Search Parameters:
  Region: Bavarian region (Central Europe with CORINE coverage)
  Bounding Box: (12.5, 60) to (17.5, 63.5)
  Date Range: 2018-03-01 to 2018-10-31
  Max Cloud Cover: 30%

Note: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).
Your search will find all tiles intersecting this region.


## Search Sentinel-2 Collection

Query the Copernicus Dataspace catalog for Sentinel-2 Level 2A imagery matching our criteria.

In [7]:
def search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover):
    """
    Search for Sentinel-2 L2A products in Copernicus Dataspace
    """
    # Build OData filter query
    filters = [
        f"Collection/Name eq 'SENTINEL-2'",
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')",
        f"ContentDate/Start gt {start_date}",
        f"ContentDate/Start lt {end_date}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{roi_polygon}')",
        f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {max_cloud_cover})"
    ]
    
    filter_query = " and ".join(filters)
    
    params = {
        "$filter": filter_query,
        "$orderby": "ContentDate/Start asc",
        "$top": 1000  # Increased to get full date range across all tiles
    }
    
    try:
        response = requests.get(SEARCH_URL, params=params, timeout=60)
        response.raise_for_status()
        results = response.json()
        return results.get('value', [])
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return []

print("✓ Search function defined")

✓ Search function defined


In [8]:
print("Searching for Sentinel-2 products...")
products = search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover)

print(f"\n✓ Found {len(products)} Sentinel-2 L2A products")
print(f"✓ All products have <{max_cloud_cover}% cloud cover (filtered server-side)")
print(f"\nThese products span multiple MGRS tiles over your region.")
print(f"Select one MGRS tile and download ~4 acquisitions.\n")
print(f"First 5 products:")
for i, product in enumerate(products[:5]):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    
    print(f"  {i+1}. {name}")
    print(f"     Date: {date}, Size: {size:.2f} GB")

Searching for Sentinel-2 products...

✓ Found 1000 Sentinel-2 L2A products
✓ All products have <30% cloud cover (filtered server-side)

These products span multiple MGRS tiles over your region.
Select one MGRS tile and download ~4 acquisitions.

First 5 products:
  1. S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.20 GB
  2. S2B_MSIL2A_20180301T103019_N0500_R108_T32VPQ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.12 GB
  3. S2B_MSIL2A_20180301T103019_N0500_R108_T33VWJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.18 GB
  4. S2B_MSIL2A_20180301T103019_N0500_R108_T33VXL_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.17 GB
  5. S2B_MSIL2A_20180301T103019_N0500_R108_T33VVJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.16 GB


## Group Products by MGRS Tile

Organize products by MGRS tile to ensure we download multiple acquisitions of the same tile.

In [9]:
# Group products by MGRS tile
from collections import defaultdict

tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    # Extract MGRS tile from product name (e.g., T32UPD from S2A_MSIL2A_..._T32UPD_...)
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018

In [10]:
# Group products by MGRS tile
from collections import defaultdict

tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    # Extract MGRS tile from product name (e.g., T32UPD from S2A_MSIL2A_..._T32UPD_...)
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")
# Select a tile to work with (choose the one with most acquisitions, or specify manually)
# Option 1: Automatic - select tile with most acquisitions
selected_tile = max(tiles.items(), key=lambda x: len(x[1]))[0] if tiles else None

# Option 2: Manual selection - uncomment and specify tile ID
# selected_tile = "T32UPD"  # Replace with your chosen tile

if selected_tile:
    tile_products = tiles[selected_tile]
    num_acquisitions = len(tile_products)
    
    print(f"Selected MGRS Tile: {selected_tile}")
    print(f"Total acquisitions available: {num_acquisitions}")
    
    # Select 4 evenly spaced acquisitions for temporal diversity
    num_to_select = 4
    if num_acquisitions >= num_to_select:
        # Calculate indices for evenly spaced selection
        indices = [int(i * (num_acquisitions - 1) / (num_to_select - 1)) for i in range(num_to_select)]
        selected_products = [tile_products[i] for i in indices]
    else:
        # If fewer than 4 acquisitions, use all of them
        selected_products = tile_products
        indices = list(range(len(tile_products)))
    
    print(f"\nSelected {len(selected_products)} evenly-spaced acquisitions for temporal diversity:")
    print("=" * 70)
    for i, (idx, product) in enumerate(zip(indices, selected_products)):
        name = product.get('Name', 'Unknown')
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. [{idx+1}/{num_acquisitions}] {date} - {size:.2f} GB")
        print(f"      {name}")
    print("=" * 70)
    print("\nThese acquisitions span the full date range for better training data diversity.")
else:
    print("❌ No tiles found. Adjust your search parameters.")
    selected_products = []

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018

## Download Sentinel-2 Products

### Download function

In [11]:
def download_product(product, output_dir, access_token):
    """
    Download a Sentinel-2 product from Copernicus Dataspace
    
    Parameters:
    -----------
    product : dict
        Product metadata from search results
    output_dir : str
        Directory to save downloaded file
    access_token : str
        OAuth2 access token
    
    Returns:
    --------
    str : Path to downloaded file, or None if failed
    """
    product_id = product['Id']
    product_name = product['Name']
    
    # Create download directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    # Check if already downloaded
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    # Build download URL
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    headers = {"Authorization": f"Bearer {access_token}"}
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        # Stream download with progress
        with requests.get(download_url, headers=headers, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192
            downloaded = 0
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=block_size):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        # Print progress every 100 MB
                        if downloaded % (100 * 1024 * 1024) < block_size:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}% ({downloaded / (1024**3):.2f} GB)")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        # Clean up partial download
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

print("✓ Download function defined")

✓ Download function defined


### Download A Selected Acquisition to scratch directory for team1

In [11]:
# Define download directory (modify for your HPC storage)
download_dir = f"/p/scratch/training2600/team1/data"

# Download the selected product
if products and access_token:
    downloaded_file = download_product(products[0], download_dir, access_token)
    if downloaded_file:
        print(f"\n✓ Product saved to: {downloaded_file}")
else:
    print("❌ Cannot download: No products found or authentication failed")

Downloading: S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
  Size: 0.20 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(022bc2b8-9288-44b4-8627-b6f79a67f09a)/$value


### Download Multiple Acquisitions to scratch directory for team1

In [13]:
# Download the evenly-spaced acquisitions from the selected tile
download_dir = f"/p/scratch/training2600/team1/data"

if selected_tile and access_token and selected_products:
    print(f"Downloading {len(selected_products)} evenly-spaced acquisitions from tile {selected_tile}...")
    print("=" * 60)
    
    for i, product in enumerate(selected_products):
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        print(f"\n--- Downloading {i+1}/{len(selected_products)} ({date}) ---")
        download_product(product, download_dir, access_token)
    
    print("\n" + "=" * 60)
    print(f"✓ Downloaded {len(selected_products)} acquisitions from tile {selected_tile}")
    print(f"✓ Acquisitions are evenly spaced across the time range")
    print(f"✓ All files saved to: {download_dir}")
else:
    print("❌ Cannot download: No tile/products selected or authentication failed")


--- Downloading 1/4 (2018-03-14) ---
Downloading: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  Size: 0.10 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(432e0e5d-85fa-4906-ac98-7008973d6db1)/$value

--- Downloading 2/4 (2018-05-09) ---
Downloading: S2A_MSIL2A_20180509T101031_N0500_R022_T33VWH_20230825T064921.SAFE
  Size: 0.15 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(2ff2313b-c53a-475a-9da0-f38157f8c61c)/$value

--- Downloading 3/4 (2018-06-12) ---
Downloading: S2B_MSIL2A_20180612T104019_N0500_R008_T33VWH_20230716T012327.SAFE
  Size: 0.07 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(77a9f78d-6b87-4f1c-b07b-9a40fa39945b)/$value

--- Downloading 4/4 (2018-08-11) ---
Downloading: S2B_MSIL2A_20180811T104019_N0500_R008_T33VWH_20230711T150315.SAFE
 

## Alternative Download: Session Based

In [14]:
def create_authenticated_session(username, password):
    """
    Create a session with authentication for downloads
    """
    session = requests.Session()
    
    # Get access token
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    
    response = session.post(AUTH_URL, data=data, timeout=30)
    response.raise_for_status()
    
    token = response.json()["access_token"]
    session.headers.update({"Authorization": f"Bearer {token}"})
    
    return session

def download_with_session(product, output_dir, session):
    """
    Download using authenticated session
    """
    product_id = product['Id']
    product_name = product['Name']
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        with session.get(download_url, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            
            os.makedirs(output_dir, exist_ok=True)
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        if downloaded % (100 * 1024 * 1024) < 8192:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}%")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

# Usage:
COPERNICUS_USERNAME = "acc6@hi.is"  # Your login email
COPERNICUS_PASSWORD = "CutiePatootie5!" # Your login password

session = create_authenticated_session(COPERNICUS_USERNAME, COPERNICUS_PASSWORD)

print(f"Downloading {len(selected_products)} evenly-spaced acquisitions from tile {selected_tile}...")
print("=" * 60)

for i, product in enumerate(selected_products):
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    print(f"\n--- Downloading {i+1}/{len(selected_products)} ({date}) ---")
    download_with_session(product, download_dir, session)

print("\n" + "=" * 60)
print(f"✓ Downloaded {len(selected_products)} acquisitions from tile {selected_tile}")
print(f"✓ Acquisitions are evenly spaced across the time range")
print(f"✓ All files saved to: {download_dir}")


--- Downloading 1/4 (2018-03-14) ---
Downloading: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  Size: 0.10 GB
  Progress: 95.5%
✓ Download complete: /p/scratch/training2600/team1/data/S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE.zip

--- Downloading 2/4 (2018-05-09) ---
Downloading: S2A_MSIL2A_20180509T101031_N0500_R022_T33VWH_20230825T064921.SAFE
  Size: 0.15 GB
  Progress: 66.8%
✓ Download complete: /p/scratch/training2600/team1/data/S2A_MSIL2A_20180509T101031_N0500_R022_T33VWH_20230825T064921.SAFE.zip

--- Downloading 3/4 (2018-06-12) ---
Downloading: S2B_MSIL2A_20180612T104019_N0500_R008_T33VWH_20230716T012327.SAFE
  Size: 0.07 GB
✓ Download complete: /p/scratch/training2600/team1/data/S2B_MSIL2A_20180612T104019_N0500_R008_T33VWH_20230716T012327.SAFE.zip

--- Downloading 4/4 (2018-08-11) ---
Downloading: S2B_MSIL2A_20180811T104019_N0500_R008_T33VWH_20230711T150315.SAFE
  Size: 0.08 GB
✓ Download complete: /p/scratch/training2600/team1/dat

## Visualisation

In [15]:
import numpy as np
import glob

def extract_and_visualize_sentinel2(zip_path, output_dir=None):
    """
    Extract a Sentinel-2 ZIP file and create an RGB visualization.
    
    Parameters:
    -----------
    zip_path : str
        Path to the Sentinel-2 ZIP file
    output_dir : str, optional
        Directory to extract to. If None, extracts to same directory as ZIP.
    
    Returns:
    --------
    str : Path to extracted SAFE directory
    """
    import zipfile
    import rasterio
    from rasterio.plot import show
    
    if output_dir is None:
        output_dir = os.path.dirname(zip_path)
    
    # Extract ZIP file
    print(f"Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the SAFE directory name (first item in the archive)
        safe_dir = zip_ref.namelist()[0].split('/')[0]
        safe_path = os.path.join(output_dir, safe_dir)
        
        if os.path.exists(safe_path):
            print(f"⚠ Already extracted: {safe_dir}")
        else:
            zip_ref.extractall(output_dir)
            print(f"✓ Extracted to: {safe_path}")
    
    return safe_path


def visualize_sentinel2_rgb(safe_path, figsize=(12, 12)):
    """
    Create an RGB visualization from Sentinel-2 SAFE directory.
    
    Uses bands B04 (Red), B03 (Green), B02 (Blue) at 10m resolution.
    
    Parameters:
    -----------
    safe_path : str
        Path to the extracted .SAFE directory
    figsize : tuple
        Figure size for the plot
    """
    import rasterio
    from rasterio.enums import Resampling
    
    # Find the 10m resolution bands (B02, B03, B04)
    # Path pattern: .SAFE/GRANULE/*/IMG_DATA/R10m/*_B0X_10m.jp2
    granule_path = os.path.join(safe_path, 'GRANULE')
    
    if not os.path.exists(granule_path):
        print(f"❌ GRANULE directory not found in {safe_path}")
        return
    
    # Get the tile subdirectory
    tile_dirs = [d for d in os.listdir(granule_path) if os.path.isdir(os.path.join(granule_path, d))]
    if not tile_dirs:
        print("❌ No tile directories found")
        return
    
    tile_dir = os.path.join(granule_path, tile_dirs[0])
    
    # Try R10m directory first (newer format), then IMG_DATA (older format)
    r10m_path = os.path.join(tile_dir, 'IMG_DATA', 'R10m')
    img_data_path = os.path.join(tile_dir, 'IMG_DATA')
    
    if os.path.exists(r10m_path):
        band_dir = r10m_path
        band_pattern = '*_B0{}_10m.jp2'
    else:
        band_dir = img_data_path
        band_pattern = '*_B0{}.jp2'
    
    # Find band files
    bands = {}
    for band_num in ['2', '3', '4']:
        pattern = os.path.join(band_dir, band_pattern.format(band_num))
        matches = glob.glob(pattern)
        if matches:
            bands[f'B0{band_num}'] = matches[0]
        else:
            # Try alternative pattern for older format
            alt_pattern = os.path.join(band_dir, f'*B0{band_num}*.jp2')
            alt_matches = glob.glob(alt_pattern)
            if alt_matches:
                bands[f'B0{band_num}'] = alt_matches[0]
    
    if len(bands) < 3:
        print(f"❌ Could not find all RGB bands. Found: {list(bands.keys())}")
        print(f"   Searched in: {band_dir}")
        return
    
    print(f"✓ Found bands: {list(bands.keys())}")
    
    # Read bands with downsampling for visualization (full resolution can be huge)
    downsample_factor = 10  # Read at 1/10 resolution for faster display
    
    rgb_bands = []
    for band_name in ['B04', 'B03', 'B02']:  # RGB order
        with rasterio.open(bands[band_name]) as src:
            # Calculate new dimensions
            new_height = src.height // downsample_factor
            new_width = src.width // downsample_factor
            
            # Read with resampling
            band_data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.average
            )
            rgb_bands.append(band_data)
    
    # Stack into RGB array
    rgb = np.stack(rgb_bands, axis=-1)
    
    # Normalize for visualization (typical Sentinel-2 values range 0-10000)
    # Use percentile-based stretching for better visualization
    p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
    rgb_normalized = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # RGB composite
    axes[0].imshow(rgb_normalized)
    axes[0].set_title('Sentinel-2 RGB Composite (B4-B3-B2)', fontsize=12)
    axes[0].axis('off')
    
    # False color (NIR-Red-Green) if available
    # For now, show a histogram of the data
    axes[1].hist(rgb[:,:,0].flatten()[::100], bins=50, alpha=0.7, label='Red (B04)', color='red')
    axes[1].hist(rgb[:,:,1].flatten()[::100], bins=50, alpha=0.7, label='Green (B03)', color='green')
    axes[1].hist(rgb[:,:,2].flatten()[::100], bins=50, alpha=0.7, label='Blue (B02)', color='blue')
    axes[1].set_xlabel('Reflectance Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Band Value Distribution', fontsize=12)
    axes[1].legend()
    axes[1].set_xlim(0, 5000)
    
    plt.tight_layout()
    plt.savefig(os.path.join(os.environ['PROJECT_training2600'],  f"{os.environ['USER']}/results/S2_vis.png"))
    
    # Print some metadata
    with rasterio.open(bands['B04']) as src:
        print(f"\nImage Metadata:")
        print(f"  Original size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
    
    return rgb_normalized

print("✓ Visualization functions defined")

✓ Visualization functions defined


In [16]:
# Visualize the first downloaded acquisition
# Adjust the path to match your download location

def is_valid_zipfile(filepath):
    """Check if a file is a valid ZIP file."""
    try:
        with zipfile.ZipFile(filepath, 'r') as zf:
            return zf.testzip() is None
    except (zipfile.BadZipFile, Exception):
        return False

if selected_products and access_token:
    # Get the first downloaded product
    first_product = selected_products[0]
    product_name = first_product.get('Name', '')
    
    # Build possible paths - handle various naming conventions
    # The product name might already end with .SAFE
    base_name = product_name.rstrip('.SAFE') if product_name.endswith('.SAFE') else product_name
    
    possible_paths = [
        os.path.join(download_dir, f"{product_name}"),           # Direct name (if it's a directory)
        os.path.join(download_dir, f"{product_name}.SAFE"),      # With .SAFE suffix
        os.path.join(download_dir, f"{base_name}.SAFE"),         # Base name with .SAFE
        os.path.join(download_dir, f"{product_name}.zip"),       # With .zip suffix  
        os.path.join(download_dir, f"{base_name}.zip"),          # Base name with .zip
    ]
    
    safe_path = None
    zip_path = None
    
    # Find the first existing path
    for path in possible_paths:
        if os.path.exists(path):
            if os.path.isdir(path):
                # It's a directory (SAFE format)
                safe_path = path
                print(f"✓ Found SAFE directory: {os.path.basename(path)}")
                break
            elif path.endswith('.zip') and is_valid_zipfile(path):
                # It's a valid ZIP file
                zip_path = path
                print(f"✓ Found valid ZIP file: {os.path.basename(path)}")
                break
            elif path.endswith('.zip'):
                # File exists but is not a valid ZIP - might be misnamed SAFE dir
                print(f"⚠ Found {os.path.basename(path)} but it's not a valid ZIP file")
                print("  Checking if it might be a SAFE directory saved with wrong extension...")
                # Check if there's a SAFE directory with similar name
                continue
    
    # If we found a ZIP file, extract it
    if zip_path and not safe_path:
        print(f"\nVisualizing: {os.path.basename(zip_path)}")
        print("=" * 60)
        safe_path = extract_and_visualize_sentinel2(zip_path)
    
    # Visualize if we have a SAFE path
    if safe_path and os.path.isdir(safe_path):
        print(f"\nVisualizing: {os.path.basename(safe_path)}")
        print("=" * 60)
        
        # Create RGB visualization directly from SAFE directory
        print("\nCreating RGB visualization...")
        visualize_sentinel2_rgb(safe_path)
        
        print("\n" + "=" * 60)
        print("✓ Visualization complete!")
        print("  - RGB composite shows the natural color view")
        print("  - Histogram shows the distribution of reflectance values")
        print("  - Data is ready for preprocessing in Lab 3.2")
    else:
        # Last resort: try to find any .SAFE directory in download_dir
        safe_dirs = glob.glob(os.path.join(download_dir, "*.SAFE"))
        if safe_dirs:
            safe_path = safe_dirs[0]
            print(f"\nFound SAFE directory: {os.path.basename(safe_path)}")
            print("=" * 60)
            
            print("\nCreating RGB visualization...")
            visualize_sentinel2_rgb(safe_path)
            
            print("\n" + "=" * 60)
            print("✓ Visualization complete!")
        else:
            print(f"\n❌ No valid Sentinel-2 data found in: {download_dir}")
            print(f"\nSearched for:")
            for p in possible_paths[:3]:
                print(f"  - {p}")
            print("\nTip: Check what files exist in your download directory:")
            print(f"  ls -la {download_dir}")
else:
    print("❌ No products selected or authentication failed")
    print("   Run the search and download cells first.")

✓ Found valid ZIP file: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE.zip

Visualizing: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE.zip
Extracting: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE.zip
✓ Extracted to: /p/scratch/training2600/team1/data/S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE

Visualizing: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE

Creating RGB visualization...
✓ Found bands: ['B02', 'B03', 'B04']

Image Metadata:
  Original size: 10980 x 10980 pixels
  Resolution: 10.0m x 10.0m
  CRS: EPSG:32633
  Bounds: BoundingBox(left=499980.0, bottom=6690240.0, right=609780.0, top=6800040.0)

✓ Visualization complete!
  - RGB composite shows the natural color view
  - Histogram shows the distribution of reflectance values
  - Data is ready for preprocessing in Lab 3.2
